# Peter results
Please note that this code is intended to be run with python 3.13

In [ ]:
import pymongo as pm
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import re

## Connect to db

In [ ]:
client = pm.MongoClient()   #We are using local host
db = client["experiment"]

##  Analysis

### Fetch data

In [ ]:
full_name = []
single_letter = []
abbreviation = []
semantic = []
syntatic = []
forms = []
abbreviation_regex = "abbrev.html$"
full_name_regex = "normal.html$"
single_letter_regex = "single.html$"
semantic_regex = "..\.semantic.."
syntatic_regex = "..\.syntactic.."


sessions = db["sessions"]
for session in sessions.find({"done": True}):
    forms.append(session["forms"])
    for trial in session["trials"]:
        # print(trial)
        # print((trial["file"]))
        if re.search(pattern=full_name_regex, string=trial["file"]):
            full_name.append(trial)
        elif re.search(pattern=abbreviation_regex, string=trial["file"]):
            abbreviation.append(trial)
        elif re.search(pattern=single_letter_regex, string=trial["file"]):
            single_letter.append(trial)
        if re.search(pattern=semantic_regex, string=trial["file"]):
            semantic.append(trial)
        elif re.search(pattern=syntatic_regex, string=trial["file"]):
            syntatic.append(trial)

### Update trials with indicators for type

In [ ]:
def update_trials_with_indecators(collection_name) -> None:
    abbreviation_regex = "abbrev.html$"
    full_name_regex = "normal.html$"
    single_letter_regex = "single.html$"
    semantic_regex = "..\.semantic.."
    syntatic_regex = "..\.syntactic.."

    collection = db[collection_name]
    for session in collection.find({"done": True}):
        for trial in session["trials"]:
            variable_type = ""
            challenge_type = ""
            if re.search(pattern=full_name_regex, string=trial["file"]):
                variable_type = "Full variable name"
            elif re.search(pattern=abbreviation_regex, string=trial["file"]):
                variable_type = "Abbreviated variable name"
            elif re.search(pattern=single_letter_regex, string=trial["file"]):
                variable_type = "Single letter variable name"
            
            if re.search(pattern=semantic_regex, string=trial["file"]):
                challenge_type = "Semantic error"
            elif re.search(pattern=syntatic_regex, string=trial["file"]):
                challenge_type = "Syntatic error"

            filter = {"USER_SESSION_ID": session["USER_SESSION_ID"], "trials.created": trial["created"]}
            operation = {"$set":  {'trials.$.variable_type': str(variable_type)}}
            ops = {"$set": {'trials.$.challenge_type': str(challenge_type)}}
            db.sessions.update_one(filter, operation)
            db.sessions.update_one(filter, ops)

In [ ]:
update_trials_with_indecators("sessions")

### Begin analysis

In [ ]:
def success_rate(bucket: list) -> [list, list]:
    success = 0
    fail = 0
    for drop in bucket:
        if drop["state"] == "success":
            success += 1
        elif drop["state"] == "failure":
            fail += 1
    return [[success, fail], ["success", "failure"]]

In [ ]:
palette_color = sns.color_palette('bright')
figure, axes = plt.subplots(1, 3)

# Creating chart for full names
data, keys = success_rate(full_name)
axes[0].pie(data, autopct='%.0f%%')
axes[0].set_title("Full names")

# Creating chart for abbreviations
data, keys = success_rate(abbreviation)
axes[1].pie(data, autopct='%.0f%%')
axes[1].set_title("Abbreviated names")

# Creating chart for single letter names
data, keys = success_rate(single_letter)
axes[2].pie(data, autopct='%.0f%%')
axes[2].set_title("Single letter names")

# Setup figure specific information
figure.tight_layout()
figure.legend(labels=keys)
figure.suptitle("Comparison between different variable names")
plt.show()

figure.savefig("./figures/Variable-names.png", bbox_inches='tight')

In [ ]:
palette_color = sns.color_palette('bright')
figure, axes = plt.subplots(1, 2)

# Creating chart for full names
data, keys = success_rate(semantic)
# axes[0].pie(full_name_DF["state"])
axes[0].pie(data, autopct='%.0f%%')
axes[0].set_title("Semantic problem")

# Creating chart for abbreviations
data, keys = success_rate(syntatic)
axes[1].pie(data, autopct='%.0f%%')
axes[1].set_title("Syntatic problem")

# Setup figure specific information
figure.tight_layout()
figure.legend(labels=keys)
figure.suptitle("Comparison between problem types")
plt.show()

figure.savefig("./figures/Problem-type.png", bbox_inches='tight')

## Analysis of questions

In [ ]:
SKIP_EMPTY_DESCRIPTIONS = True
trials_DF = pd.DataFrame(full_name + abbreviation + single_letter)
output_string_MD = "|Description | State| Type|\n|--|--|--|"
for index, trial in trials_DF.iterrows():
    if (trial["description"] == "" and SKIP_EMPTY_DESCRIPTIONS):
        continue
    output_string_MD += f"\n|{trial["description"]}|{trial["state"]}|{trial["variable_type"]}|"
print(output_string_MD)

|Description | State| Type|
|--|--|--|
||success|Full variable name|
||success|Full variable name|
|Should increase count of that element by one, not by index|success|Full variable name|
|- instead of queals|success|Full variable name|
|The frequency should only increment by one.|success|Full variable name|
||success|Full variable name|
||success|Full variable name|
||success|Full variable name|
|Index should not be used as frequency increment|success|Full variable name|
|Missing equals?|success|Full variable name|
|not the right array|success|Full variable name|
|invalid syntax, have to have eaqual to, not minus|success|Full variable name|
|Wrongly uses the part list, which does contain the expected value and does not change for each iteration in the loop.|success|Full variable name|
|There is a '-' sign, instead of a '=' sign, making us never assign the value to personage, even if we could compile. Right now we are not assign person age to something but it appears we are trying to use it, making the compiler mad at us.|success|Full variable name|
||success|nan|
||failure|nan|
||success|Abbreviated variable name|
||success|Abbreviated variable name|
||success|Abbreviated variable name|
|[elapsed]|elapsed|Abbreviated variable name|
|Missing semicolon, rookie mistake (I frequently still do it)|success|Abbreviated variable name|
|The appending must be offset by the length of the first string|success|Abbreviated variable name|
||success|Abbreviated variable name|
||failure|Abbreviated variable name|
||success|Abbreviated variable name|
|concatenation was done wrong|success|Abbreviated variable name|
|Should be list return type|failure|Abbreviated variable name|
|if we add to index+1 we will overwrite that positions in the next loop|success|Abbreviated variable name|
|wrong character|success|Abbreviated variable name|
|The frequency is using the index, which means if we were on letter 2 it would increment 2, letter 4 would increase the value 4 times. This is not how frequency works, as it should increment one time per occurrence.|success|Abbreviated variable name|
|Does not use the same starting sign '[' as the end sign ')'. This appears to be a function call. so '(' should be used.|success|Abbreviated variable name|
||success|nan|
||success|nan|
||success|Single letter variable name|
||success|Single letter variable name|
|should be the second element of f not c|success|Single letter variable name|
|Should be paranthesees instead of bracket|success|Single letter variable name|
|Accessing the wrong variable.|success|Single letter variable name|
||success|Single letter variable name|
||success|Single letter variable name|
||success|Single letter variable name|
|Unsure, but this might not be a method for dictionaries|failure|Single letter variable name|
|[ should be (|success|Single letter variable name|
|increase the frequency with 1, not with the index number|success|Single letter variable name|
|missing semicolon|success|Single letter variable name|
|We are not zipping two lists, we are appending one list to the other. This code will not only write over previous values, it will not correctly append the list in the first place.|success|Single letter variable name|
|Missing a semicolon|success|Single letter variable name|
||success|nan|
||success|nan|

|Description | State| Type|
|--|--|--|
|Should increase count of that element by one, not by index|success|Full variable name|
|- instead of queals|success|Full variable name|
|The frequency should only increment by one.|success|Full variable name|
|Index should not be used as frequency increment|success|Full variable name|
|Missing equals?|success|Full variable name|
|not the right array|success|Full variable name|
|invalid syntax, have to have eaqual to, not minus|success|Full variable name|
|Wrongly uses the part list, which does contain the expected value and does not change for each iteration in the loop.|success|Full variable name|
|There is a '-' sign, instead of a '=' sign, making us never assign the value to personage, even if we could compile. Right now we are not assign person age to something but it appears we are trying to use it, making the compiler mad at us.|success|Full variable name|
|[elapsed]|elapsed|Abbreviated variable name|
|Missing semicolon, rookie mistake (I frequently still do it)|success|Abbreviated variable name|
|The appending must be offset by the length of the first string|success|Abbreviated variable name|
|concatenation was done wrong|success|Abbreviated variable name|
|Should be list return type|failure|Abbreviated variable name|
|if we add to index+1 we will overwrite that positions in the next loop|success|Abbreviated variable name|
|wrong character|success|Abbreviated variable name|
|The frequency is using the index, which means if we were on letter 2 it would increment 2, letter 4 would increase the value 4 times. This is not how frequency works, as it should increment one time per occurrence.|success|Abbreviated variable name|
|Does not use the same starting sign '[' as the end sign ')'. This appears to be a function call. so '(' should be used.|success|Abbreviated variable name|
|should be the second element of f not c|success|Single letter variable name|
|Should be paranthesees instead of bracket|success|Single letter variable name|
|Accessing the wrong variable.|success|Single letter variable name|
|Unsure, but this might not be a method for dictionaries|failure|Single letter variable name|
|[ should be (|success|Single letter variable name|
|increase the frequency with 1, not with the index number|success|Single letter variable name|
|missing semicolon|success|Single letter variable name|
|We are not zipping two lists, we are appending one list to the other. This code will not only write over previous values, it will not correctly append the list in the first place.|success|Single letter variable name|
|Missing a semicolon|success|Single letter variable name|

### Getting language proficiency

In [ ]:
language_forms = []
keys = list(forms[1][1].keys())
for i in forms:
    for j in i:
        if keys == list(j.keys()):
            language_forms.append(j)

language_forms_DF = pd.DataFrame(language_forms)
language_forms_DF

In [ ]:
sns.set_style("whitegrid")
languages = ["fsharp", "java", "js", "c_or_cpp", "csharp", "php", "vbnet"]
melting_pot= pd.melt(language_forms_DF, value_vars=languages)

graph = sns.barplot(data=melting_pot, 
                    x="variable", 
                    y="value",
                    hue="variable",
                    palette=sns.color_palette("husl", 7)
                    )
graph.invert_yaxis()
graph.set_yticks([0,1,2,3,4,5])
graph.set_yticklabels([0,1,2,3,4,5])

plt.xlabel("")
plt.ylabel("Experience scale")
plt.title("Language proficiency")

plt.savefig("./figures/Language-proficiency.png", bbox_inches='tight')

In [ ]:
lang_amount = {}
for str in language_forms_DF.other_languages:
    for lang in str.split(", "):
        if lang in list(lang_amount.keys()):
            lang_amount[lang] += 1
        else:
            lang_amount[lang] = 1
lang_amount